In [1]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.4 MB/s eta 0:00:00 0:00:01


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import sentencepiece as spm

import math
import os
import re
import urllib.request
import zipfile
import numpy as np
import matplotlib.pyplot as plt

### **변경해야할 것**
- Encoder 삭제
- Encoder-Decoder Attention 삭제
- Positional Encoding - > Positional Embedding으로 변경
- input 에 enc 와 같이 있을 때는 dec 만 이용하도록 변경

| 구분 | 기본 Transformer | Decoder-Only (GPT) | 변경 내용 |
|------|------------------|-------------------|----------|
| 전체 구조 | Encoder + Decoder | Decoder Only | Encoder 완전히 제거 |
| Encoder | 사용함 | ❌ 제거 | Encoder 관련 코드 삭제 |
| Encoder-Decoder Attention | 있음 | ❌ 없음 | cross-attention 제거 |
| Self-Attention | 있음 | 있음 (Masked) | look-ahead mask 필수 |
| Positional Encoding | Sin/Cos | Learnable Embedding | nn.Embedding으로 변경 |
| 입력 구조 | (enc_input, dec_input) | 단일 입력 (dec_input) | 하나의 시퀀스로 통합 |

### **Positional Encoding**

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, position, d_model):
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model
        self.position = position

        self.pos_encoding = self._build_pos_encoding(position, d_model)

    def _get_angles(self, position, i, d_model):
        return 1.0 / (10000.0 ** ((2.0 * (i // 2)) / d_model)) * position

    def _build_pos_encoding(self, position, d_model):
        pos = torch.arange(position, dtype=torch.float32).unsqueeze(1)
        i = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)

        angle_rads = self._get_angles(pos, i, d_model)
        sines = torch.sin(angle_rads[:, 0::2])
        cosines = torch.cos(angle_rads[:, 1::2])

        pos_encoding = torch.zeros(position, d_model)
        pos_encoding[:, 0::2] = sines
        pos_encoding[:, 1::2] = cosines

        pos_encoding = pos_encoding.unsqueeze(0)  # shape: [1, position, d_model]
        return pos_encoding

    def forward(self, x):
        return x + self.pos_encoding[:, :x.size(1), :].to(x.device)

### **Positional Embedding Matrix**

In [ ]:
# class PositionalEncoding(nn.Module):
#     def __init__(self, position, d_model):
#         super(PositionalEncoding, self).__init__()
#         self.d_model = d_model
#         self.position = position

#         self.pos_encoding = self._build_pos_encoding(position, d_model)

#     def _get_angles(self, position, i, d_model):
#         return 1.0 / (10000.0 ** ((2.0 * (i // 2)) / d_model)) * position

#     def _build_pos_encoding(self, position, d_model):
#         pos = torch.arange(position, dtype=torch.float32).unsqueeze(1)
#         i = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)

#         angle_rads = self._get_angles(pos, i, d_model)
#         sines = torch.sin(angle_rads[:, 0::2])
#         cosines = torch.cos(angle_rads[:, 1::2])

#         pos_encoding = torch.zeros(position, d_model)
#         pos_encoding[:, 0::2] = sines
#         pos_encoding[:, 1::2] = cosines

#         pos_encoding = pos_encoding.unsqueeze(0)  # shape: [1, position, d_model]
#         return pos_encoding

#     def forward(self, x):
#         return x + self.pos_encoding[:, :x.size(1), :].to(x.device)

### **Scaled Dot Product**

In [4]:
def scaled_dot_product_attention(query, key, value, mask=None):

    # 1) Q와 K의 내적을 통해 score(유사도) 계산
    # key.transpose(-1, -2): (batch_size, heads, depth, seq_len)
    # matmul 결과 shape: (batch_size, heads, seq_len, seq_len)
    matmul_qk = torch.matmul(query, key.transpose(-1, -2))

    # 2) depth에 따라 정규화
    depth = key.size(-1)  # depth = d_model / heads
    logits = matmul_qk / math.sqrt(depth)

    # 3) 마스크가 주어졌다면 -1e9(아주 작은 값)를 더해 소프트맥스에서 제외시키도록 함
    if mask is not None:
        # 텐서플로우: logits += (mask * -1e9)
        # 파이토치 동일 적용
        logits = logits + (mask * -1e9)

    # 4) 소프트맥스 계산해 attention weights 생성
    attention_weights = F.softmax(logits, dim=-1)

    # 5) attention weights와 value의 내적
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

### **MultiHeadAttention**

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, name="multi_head_attention"):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model은 num_heads로 나누어떨어져야 함
        assert d_model % num_heads == 0

        self.depth = d_model // num_heads

        # 파이토치에서 Dense는 nn.Linear로 대응
        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)

        self.out_dense = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        """
        x: (batch_size, seq_len, d_model)
        => (batch_size, num_heads, seq_len, depth) 형태로 변환
        """
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        x = x.permute(0, 2, 1, 3)  # (batch_size, num_heads, seq_len, depth)
        return x

    def forward(self, query, key, value, mask=None):
        """
        query, key, value: (batch_size, seq_len, d_model)
        mask: (batch_size, 1, seq_len, seq_len) 등으로 broadcast 가능하도록 구성
        """
        batch_size = query.size(0)

        # Q, K, V에 각각 Linear 적용
        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        # Head 분할
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        # 스케일드 닷 프로덕트 어텐션
        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)

        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()

        # 다시 (batch_size, seq_len, d_model)로 합치기
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        # 최종 Dense
        output = self.out_dense(concat_attention)
        return output


### **Padding**

In [6]:
def create_padding_mask(x):
    # x == 0 위치를 찾아 float형 1로 변환
    mask = (x == 0).float()
    # (batch_size, seq_len) -> (batch_size, 1, 1, seq_len)
    mask = mask.unsqueeze(1).unsqueeze(2)
    return mask


def create_look_ahead_mask(x):
    seq_len = x.size(1)

    # (seq_len, seq_len) 크기의 하삼각 행렬(tril) 생성 후 1에서 빼서
    # 상삼각이 1, 하삼각(자기 자신 포함)이 0이 되도록 설정
    # => 미래 토큰(자신 인덱스보다 큰 위치) 마스킹
    look_ahead_mask = 1 - torch.tril(torch.ones((seq_len, seq_len)))

    # 패딩 마스크 생성 (shape: (batch_size, 1, 1, seq_len))
    padding_mask = create_padding_mask(x)

    # look_ahead_mask: (seq_len, seq_len) -> (1, seq_len, seq_len)
    look_ahead_mask = look_ahead_mask.unsqueeze(0)
    # -> (1, seq_len, seq_len) -> (1, 1, seq_len, seq_len)
    look_ahead_mask = look_ahead_mask.unsqueeze(1)
    look_ahead_mask = look_ahead_mask.to(x.device)

    # look-ahead 마스크와 패딩 마스크를 합성 (둘 중 하나라도 1이면 마스킹)
    # 최종 shape은 브로드캐스팅으로 (batch_size, 1, seq_len, seq_len)
    combined_mask = torch.max(look_ahead_mask, padding_mask)
    return combined_mask

### **Encoder --> GPT : Decoder-Only model 라서 사용하지 않음**

In [7]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)  # 이전에 구현한 MHA
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)

        # 피드포워드 부분 (Dense -> ReLU -> Dense)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, mask=None):
        # (1) 멀티 헤드 어텐션 (셀프 어텐션)
        attn_output = self.mha(x, x, x, mask)  # (batch_size, seq_len, d_model)
        attn_output = self.dropout1(attn_output)
        out1 = self.norm1(x + attn_output)     # 잔차 연결 + LayerNorm

        # (2) 피드포워드 신경망
        ffn_output = self.ffn(out1)            # (batch_size, seq_len, d_model)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.norm2(out1 + ffn_output)   # 잔차 연결 + LayerNorm

        return out2


In [8]:
class Encoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,
                 ff_dim,
                 d_model,
                 num_heads,
                 dropout=0.1):
        super(Encoder, self).__init__()
        self.d_model = d_model

        # (1) 임베딩 레이어
        self.embedding = nn.Embedding(vocab_size, d_model)

        # (2) 포지셔널 인코딩
        self.pos_encoding = PositionalEncoding(position=vocab_size, d_model=d_model)

        self.dropout = nn.Dropout(dropout)

        # (3) EncoderLayer 쌓기
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        # (1) 임베딩 & sqrt(d_model)로 스케일링
        x = self.embedding(x) * math.sqrt(self.d_model)

        # (2) 포지셔널 인코딩 적용 + 드롭아웃
        x = self.pos_encoding(x)  # shape: (batch_size, seq_len, d_model)
        x = self.dropout(x)

        # (3) num_layers만큼 쌓아올린 EncoderLayer 통과
        for layer in self.enc_layers:
            x = layer(x, mask)

        return x



![nn](gpt1_architecture.png)

### **Decoder**

In [179]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(DecoderLayer, self).__init__()

        # 첫 번째 서브 레이어 (디코더 내부 셀프 어텐션)
        self.self_mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)

        ##############################################################
        ################# Encoder - Decoder Attetnion NO #############
        ##############################################################
        
        # 두 번째 서브 레이어 (인코더-디코더 어텐션)
        # self.encdec_mha = MultiHeadAttention(d_model, num_heads)
        # self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        # self.dropout2 = nn.Dropout(dropout)
        ##############################################################

        
        # 세 번째 서브 레이어 (피드포워드 네트워크)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),  # Dense(units=ff_dim)
            nn.ReLU(),                   # activation='relu'
            nn.Linear(ff_dim, d_model)   # Dense(units=d_model)
        )
        self.norm3 = nn.LayerNorm(d_model, eps=1e-6)

        # 드롭아웃
        self.dropout1 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_outputs, look_ahead_mask=None, padding_mask=None):
        # 1) 셀프 어텐션 (디코더 내부)
        self_attn_out = self.self_mha(x, x, x, mask=look_ahead_mask)
        self_attn_out = self.dropout1(self_attn_out)
        out1 = self.norm1(x + self_attn_out)  # 잔차 연결 + LayerNorm

        ##############################################################
        ################# Encoder - Decoder Attetnion NO #############
        ##############################################################
        
        # 2) 인코더-디코더 어텐션
        # encdec_attn_out = self.encdec_mha(out1, enc_outputs, enc_outputs, mask=padding_mask)
        # encdec_attn_out = self.dropout2(encdec_attn_out)
        # out2 = self.norm2(out1 + encdec_attn_out)  # 잔차 연결 + LayerNorm
        ##############################################################

        
        # 3) 피드포워드 (Dense -> ReLU -> Dense)
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout3(ffn_out)
        out3 = self.norm3(out1 + ffn_out)  # 잔차 연결 + LayerNorm

        return out3

In [227]:
class Decoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,
                 ff_dim,
                 d_model,
                 num_heads,
                 max_seq_len,
                 dropout=0.1):
        super(Decoder, self).__init__()
        self.d_model = d_model

        # (1) 임베딩 레이어
        self.embedding = nn.Embedding(vocab_size, d_model)

        # (2) 포지셔널 인코딩
        # 실제 학습 시에는 최대 시퀀스 길이에 맞추어 쓰기도 함
        # 기존 Transformer에서 사용하는 
        # self.pos_encoding = PositionalEncoding(position=vocab_size, d_model=d_model)

        ##############################################################
        ################### Positional Embedding Matrix ##############
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        ##############################################################

        self.dropout = nn.Dropout(dropout)

        # (3) DecoderLayer 쌓기
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, enc_outputs, look_ahead_mask=None, padding_mask=None):
        # (1) 임베딩 + sqrt(d_model)로 스케일링
        x = self.embedding(x) * math.sqrt(self.d_model)

        # # (2) 포지셔널 인코딩 + 드롭아웃
        # x = self.pos_encoding(x)    # (batch_size, tgt_seq_len, d_model)
        # x = self.dropout(x)

        ##############################################################
        ################### Positional Embedding Matrix ##############
        ##############################################################
        # (2) 포지셔널 embedding + 드롭아웃
        batch_size, seq_len, _ = x.size()
        position = torch.arange(0, seq_len, device=x.device)   
        position = position.unsqueeze(0).expand(batch_size, seq_len) # (1, seq_len) -> (batch_size, seq_len)
        pos = self.pos_embedding(position)  # positional embedding lookup
        x += pos
        
        x = self.dropout(x)
        
        # (3) num_layers만큼 쌓인 DecoderLayer 통과
        for layer in self.dec_layers:
            x = layer(x, enc_outputs, look_ahead_mask, padding_mask)

        return x

- encoder : 문장을 읽고 정보를 압축하는 모듈
- decoder : 압축된 정보를 바탕으로 출력 sequence를 만드는 역할을 하는 모듈
- context vecotor : 압축된 정보

In [228]:
# mkdir -p ~/work/transformer_chatbot/data/ && cd ~/work/transformer_chatbot/data/

In [229]:
!wget https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv

--2026-04-21 07:45:14--  https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv [following]
--2026-04-21 07:45:14--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv.3’

ChatbotData.csv.3   100%[===================>] 868.99K  --.-KB/s    in 0.1s    

2026-04-21 07:45:15 (5.69 MB/s) - ‘ChatbotData.csv.3’ saved [889842/889842]



## Wellness DATA 로 Transformer model 적용해보기

In [230]:
torch.manual_seed(20260421)

In [231]:
import pandas as pd
data=pd.read_csv('ChatbotData.csv')

In [232]:
data

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0
...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,2
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,2
11820,흑기사 해주는 짝남.,설렜겠어요.,2
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,2


### - 챗봇 트레이닝용 문답 페어 11,823개  
### - 일상다반사 0, 이별(부정) 1, 사랑(긍정) 2로 레이블링

In [233]:
import copy
data_origin = copy.deepcopy(data)

## **Step 2.데이터 전처리하기**

In [234]:
# 전처리 함수
def preprocess_sentence_eng(sentence):
  # 입력받은 sentence를 소문자로 변경하고 양쪽 공백을 제거
  sentence = sentence.lower().strip()

  # 단어와 구두점(punctuation) 사이의 거리를 만듭니다.
  # 예를 들어서 "I am a student." => "I am a student ."와 같이
  # student와 온점 사이에 거리를 만듭니다.
  sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
  sentence = re.sub(r'[" "]+', " ", sentence)

  # (a-z, A-Z, ".", "?", "!", ",")를 제외한 모든 문자를 공백인 ' '로 대체합니다.
  sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
  sentence = sentence.strip()
  return sentence

def preprocess_sentence_kor(sentence):
    # 양쪽 공백 제거
    sentence = sentence.strip()

    # 구두점 앞뒤에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    # 여러 공백 하나로
    sentence = re.sub(r"\s+", " ", sentence)

    # 한글, 영어, 숫자, 구두점만 남기기
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)

    # 다시 공백 정리
    sentence = sentence.strip()
    
    return sentence

In [235]:
data['Q'] = data['Q'].apply(preprocess_sentence_kor)
data['A'] = data['A'].apply(preprocess_sentence_kor)

In [236]:
data

,Q,A,label
0,12시 땡 !,하루가 또 가네요 .,0
1,1지망 학교 떨어졌어,위로해 드립니다 .,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠 .,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠 .,0
4,PPL 심하네,눈살이 찌푸려지죠 .,0
...,...,...,...
11818,훔쳐보는 것도 눈치 보임 .,티가 나니까 눈치가 보이는 거죠 !,2
11819,훔쳐보는 것도 눈치 보임 .,훔쳐보는 거 티나나봐요 .,2
11820,흑기사 해주는 짝남 .,설렜겠어요 .,2
11821,힘든 연애 좋은 연애라는게 무슨 차이일까 ?,잘 헤어질 수 있는 사이 여부인 거 같아요 .,2


In [237]:
lines = []

for q, a in zip(data['Q'], data['A']):
    text = f"Context: {q} Answer: {a}"
    lines.append(text)

with open("gpt_corpus.txt", "w", encoding="utf-8") as f:
    for line in lines:
        f.write(line + "\n")

In [238]:
lines[:5]

['Context: 12시 땡 ! Answer: 하루가 또 가네요 .',
 'Context: 1지망 학교 떨어졌어 Answer: 위로해 드립니다 .',
 'Context: 3박4일 놀러가고 싶다 Answer: 여행은 언제나 좋죠 .',
 'Context: 3박4일 정도 놀러가고 싶다 Answer: 여행은 언제나 좋죠 .',
 'Context: PPL 심하네 Answer: 눈살이 찌푸려지죠 .']

 - 이후에 **SOS + Question + Delim + Answer + EOS** 게 처리할 예정임
 - 여기서만 Context, Answer 로 사용하고 토크나이징할 때 없앨 예정
 - spm.SentencePiece넣기 전에 문장 구조를 만들고 싶어서 잠시만 이렇게 정리해보고자 합니다


### **Pair 쌍 처리 X**

In [239]:
# pairs = list(zip(data['Q'], data['A']))

In [240]:
# pairs[:5]

In [241]:
# print('전체 샘플 수 :', len(pairs))

In [242]:
data.isnull().sum()

Q        0
A        0
label    0
dtype: int64

In [243]:
# q_lengths = [len(q.split()) for q, a in pairs]
# a_lengths = [len(a.split()) for q, a in pairs]

# plt.figure(figsize=(12, 5))

# # 질문 길이
# plt.subplot(1, 2, 1)
# plt.hist(q_lengths, bins=30)
# plt.title("Q Length Distribution")
# plt.xlabel("Length")
# plt.ylabel("Count")

# # 답변 길이
# plt.subplot(1, 2, 2)
# plt.hist(a_lengths, bins=30)
# plt.title("A Length Distribution")
# plt.xlabel("Length")
# plt.ylabel("Count")

# plt.show()

In [244]:
# print("Q 평균:", sum(q_lengths)/len(q_lengths))
# print("A 평균:", sum(a_lengths)/len(a_lengths))
# print("Q 최대:", max(q_lengths))
# print("A 최대:", max(a_lengths))

## **Step 3. StentencePiece 사용하기**

In [245]:
# corpus_file = "clean_kor_corpus.txt"
# with open(corpus_file, 'w', encoding='utf-8') as f:
#     for q, a in pairs:
#         f.write(q + "\n")
#         f.write(a + "\n")

In [246]:
spm.SentencePieceTrainer.Train(
    input="gpt_corpus.txt",
    model_prefix="spm_gpt",
    vocab_size=8000,
    character_coverage=0.9995,   
    model_type="unigram",        
    max_sentence_length=999999,
    bos_id=1,
    eos_id=2,
    pad_id=0,
    unk_id=3,

    # user_defined_symbols=["<Context>", "<Answer>"] 
    user_defined_symbols=["<sep>"]
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: gpt_corpus.txt
  input_format: 
  model_prefix: spm_gpt
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 999999
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <sep>
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable

In [247]:
sp = spm.SentencePieceProcessor()
sp.Load("spm_gpt.model")

True

In [248]:
for i in range(20):
    print(sp.id_to_piece(i))

<pad>
<s>
</s>
<unk>
<sep>
t
:
e
▁.
n
o
r
w
x
ns
▁A
▁C
▁
요
▁?


In [249]:
# 예제 문장
sentence = "안녕하세요, 만나서 반갑습니다 !"
sentence = preprocess_sentence_kor(sentence)
print("전처리 후의 문장:", sentence)

# 1. 토크나이징 (subword 단위로 분할)
tokens = sp.encode(sentence, out_type=str)
print("Tokenized:", tokens)

# 2. 인코딩 (서브워드를 정수 ID로 변환)
encoded = sp.encode(sentence, out_type=int)
print("Encoded:", encoded)

# 3. 디코딩 (정수 ID → 원본 문장 복원)
decoded = sp.decode(encoded)
print("Decoded:", decoded)

전처리 후의 문장: 안녕하세요 , 만나서 반갑습니다 !
Tokenized: ['▁안녕하세', '요', '▁,', '▁만나', '서', '▁반갑', '습니', '다', '▁!']
Encoded: [2708, 18, 238, 189, 40, 3566, 115, 48, 62]
Decoded: 안녕하세요 , 만나서 반갑습니다 !


## **Task-specific Input Trnasformation**
- ###  **SOS + Question + Delim + Answer + EOS**
- ![nn](gpt1_architecture.png)

In [290]:
class WellnessDataset(Dataset):
    def __init__(self, lines, sp, max_length=50):
        super().__init__()
        self.sp = sp
        self.max_length = max_length
        self.data = []

        for text in lines:
            q_part, a_text = text.split(" Answer:", 1)
            q_text = q_part.replace("Context:", "").strip()
            a_text = a_text.strip()

            # 1) 토크나이즈, 정수 인코딩
            q_ids = sp.EncodeAsIds(q_text)
            a_ids = sp.EncodeAsIds(a_text)


            sos_id = sp.bos_id() if sp.bos_id() >= 0 else 1  # 혹은 임의값
            eos_id = sp.eos_id() if sp.eos_id() >= 0 else 2
            sep_id = sp.piece_to_id("<sep>")
            pad_id = sp.pad_id()

            ######### pair 쌍 처리 방식 - delete ##############
            # q_tokens = [sos_id] + q_ids + [eos_id]
            # a_tokens = [sos_id] + a_ids + [eos_id]
            
            # # 4) 고정 길이 패딩
            # q_tokens += [0]*(max_length - len(q_tokens))  # 0 -> <pad> 가정
            # a_tokens += [0]*(max_length - len(a_tokens))


            
            #####################################################
            ########## Context + Answer 한 개의 문장 #############
            ######################################################
            tokens = [sos_id] + q_ids + [sep_id] +a_ids + [eos_id]
            #####################################################3
            
            # 3) 길이 제한
            if len(tokens) > max_length:
                continue

            ############## padding 추가 ############################
            tokens += [pad_id] * (max_length - len(tokens))
            ########################################################

            
            # (teacher forcing용)
            dec_input = tokens[:-1]
            target = tokens[1:]

            self.data.append({
                # "enc_input": q_tokens,
                "dec_input": dec_input,
                "target": target
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        # enc_input = torch.tensor(sample["enc_input"], dtype=torch.long)
        dec_input = torch.tensor(sample["dec_input"], dtype=torch.long)
        target = torch.tensor(sample["target"], dtype=torch.long)
        return dec_input, target

- sp 의 id 확인하기 : sos, eos, pad, unk, sep  

In [291]:
print("BOS ID:", sp.bos_id())
print("EOS ID:", sp.eos_id())
print("PAD ID:", sp.pad_id())
print("UNK ID:", sp.unk_id())

BOS ID: 1
EOS ID: 2
PAD ID: 0
UNK ID: 3


In [292]:
sep_id = sp.piece_to_id("<sep>")
print(sep_id)

4


In [293]:
# Dataset 구현하기
dataset = WellnessDataset(lines, sp, max_length=100)

In [294]:
## 전처리 잘 되었는지 확인하기
for decoder_input, decoder_label  in dataset:
    print("텐서 크기 :",decoder_input.size())
    print(decoder_input)
    print(sp.decode(decoder_input.tolist()))
    print(decoder_label)
    print(sp.decode(decoder_label.tolist()))
    break

텐서 크기 : torch.Size([99])
tensor([   1, 4231,  436, 2409,   62,    4,  280,   22,  120, 3443,   18,    8,
           2,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])
12시 땡 !<sep> 하루가 또 가네요 .
tensor([4231,  436, 2409,   62,    4,  280,   22,  120, 3443,   18,    8,    2,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,

In [295]:
# DataLoader 구성하기
dataloader = DataLoader(dataset,batch_size=32,shuffle=True)

In [296]:
dataloader

In [297]:
for text in lines[:3]:
    q_part, a_text = text.split(" Answer:", 1)
    q_text = q_part.replace("Context:", "").strip()
    a_text = a_text.strip()

    print(q_text, ":", a_text)

12시 땡 ! : 하루가 또 가네요 .
1지망 학교 떨어졌어 : 위로해 드립니다 .
3박4일 놀러가고 싶다 : 여행은 언제나 좋죠 .


In [298]:
for decoder_input, decoder_label in dataloader:
    print(decoder_input.size())
    print(decoder_label.size())
    break

torch.Size([32, 99])
torch.Size([32, 99])


In [299]:
print("Vocab size:", sp.GetPieceSize())
print("BOS:", sp.bos_id(), sp.id_to_piece(sp.bos_id()))
print("EOS:", sp.eos_id(), sp.id_to_piece(sp.eos_id()))

Vocab size: 8000
BOS: 1 <s>
EOS: 2 </s>


In [300]:
max_seq_len = 100

### **Transformer 모델 구축**

In [301]:
class Transformer(nn.Module):
    def __init__(self,
                 vocab_size,
                 num_layers,      # 인코더/디코더 층 수
                 units,           # feed-forward 네트워크의 중간 차원(ff_dim)
                 d_model,         # 임베딩 및 내부 표현 차원
                 num_heads,       # 멀티헤드 어텐션의 헤드 수
                 dropout=0.1):
        super(Transformer, self).__init__()

        ##############################################################
        ######################## Encoder NO ##########################
        ##############################################################
        # 인코더
        # self.encoder = Encoder(
        #     vocab_size=vocab_size,
        #     num_layers=num_layers,
        #     ff_dim=units,
        #     d_model=d_model,
        #     num_heads=num_heads,
        #     dropout=dropout
        # )
        ###############################################################
        
        # 디코더
        self.decoder = Decoder(
            vocab_size=vocab_size,
            num_layers=num_layers,
            ff_dim=units,
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            max_seq_len=max_seq_len,
        )

        # 최종 출력층: (d_model) -> (vocab_size)
        self.final_linear = nn.Linear(d_model, vocab_size)

        # 참고: 텐서플로우 코드의 `name="transformer"`는 파이토치에선 보통 사용 안 함

    def forward(self, dec_inputs):
        # 1) 인코더 패딩 마스크 생성
        # enc_padding_mask = create_padding_mask(inputs)     # shape (batch_size, 1, 1, src_seq_len)

        # 2) 디코더 look-ahead + 패딩 마스크
        look_ahead_mask = create_look_ahead_mask(dec_inputs)  # shape (batch_size, 1, tgt_seq_len, tgt_seq_len)

        # 3) 디코더에서 인코더 출력 쪽을 마스킹할 때 쓸 패딩 마스크
        # dec_padding_mask = create_padding_mask(inputs)        # shape (batch_size, 1, 1, src_seq_len)

        ##############################################################
        ################# Encoder - Decoder Attetnion NO #############
        ##############################################################
        # # 4) 인코더 수행
        # enc_outputs = self.encoder(
        #     x=inputs,
        #     mask=enc_padding_mask
        # )  # shape: (batch_size, src_seq_len, d_model)

        # # 5) 디코더 수행
        # dec_outputs = self.decoder(
        #     x=dec_inputs,           # (batch_size, tgt_seq_len)
        #     enc_outputs=enc_outputs,# (batch_size, src_seq_len, d_model)
        #     look_ahead_mask=look_ahead_mask,
        #     padding_mask=dec_padding_mask
        # )  # shape: (batch_size, tgt_seq_len, d_model)
        ################################################################





        
        ##############################################################
        ######################## Decoder Only ########################
        ##############################################################
        dec_outputs = self.decoder(
            x = dec_inputs,
            enc_outputs = None,
            look_ahead_mask=look_ahead_mask,
            padding_mask=None,
        )
        ###############################################################

        
        # 6) 최종 Dense (vocab_size)
        logits = self.final_linear(dec_outputs)  # (batch_size, tgt_seq_len, vocab_size)
        return logits

In [302]:
print(sp.GetPieceSize())

8000


In [303]:
# 예: 하이퍼파라미터 설정
NUM_LAYERS = 4     # 인코더/디코더 층 수
D_MODEL = 256      # 임베딩 및 내부 표현 차원
NUM_HEADS = 8      # 멀티헤드 어텐션에서의 헤드 수
UNITS = 512        # 피드포워드 신경망의 은닉 차원
DROPOUT = 0.1      # 드롭아웃 비율
VOCAB_SIZE = 8000 # 단어 집합 크기(예시)

# 모델 생성
model = Transformer(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    units=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dropout=DROPOUT,
)

print(model)

Transformer(
  (decoder): Decoder(
    (embedding): Embedding(8000, 256)
    (pos_embedding): Embedding(100, 256)
    (dropout): Dropout(p=0.1, inplace=False)
    (dec_layers): ModuleList(
      (0-3): 4 x DecoderLayer(
        (self_mha): MultiHeadAttention(
          (query_dense): Linear(in_features=256, out_features=256, bias=True)
          (key_dense): Linear(in_features=256, out_features=256, bias=True)
          (value_dense): Linear(in_features=256, out_features=256, bias=True)
          (out_dense): Linear(in_features=256, out_features=256, bias=True)
        )
        (norm1): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=256, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=256, bias=True)
        )
        (norm3): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout3): Dropout(p=0.1, 

- summary

In [318]:
!pip install torchinfo

In [319]:
from torchinfo import summary

summary(
    model,
    input_size=(32, 100),  # (batch_size, seq_len)
    dtypes=[torch.long]
)

Layer (type:depth-idx)                        Output Shape              Param #
Transformer                                   [32, 100, 8000]           --
├─Decoder: 1-1                                [32, 100, 256]            --
│    └─Embedding: 2-1                         [32, 100, 256]            2,048,000
│    └─Embedding: 2-2                         [32, 100, 256]            25,600
│    └─Dropout: 2-3                           [32, 100, 256]            --
│    └─ModuleList: 2-4                        --                        --
│    │    └─DecoderLayer: 3-1                 [32, 100, 256]            527,104
│    │    └─DecoderLayer: 3-2                 [32, 100, 256]            527,104
│    │    └─DecoderLayer: 3-3                 [32, 100, 256]            527,104
│    │    └─DecoderLayer: 3-4                 [32, 100, 256]            527,104
├─Linear: 1-2                                 [32, 100, 8000]           2,056,000
Total params: 6,238,016
Trainable params: 6,238,016
Non-t

In [304]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)

In [305]:
def train_step(model, batch, optimizer, loss_function, device):
    model.train()
    dec_input, target = [x.to(device) for x in batch]

    optimizer.zero_grad()

    # 모델 포워드 패스
    logits = model(dec_input)  # (batch_size, seq_len, vocab_size)

    # Loss 계산 (패딩 토큰 무시)
    loss = loss_function(logits.permute(0, 2, 1), target)  # (batch_size, vocab_size, seq_len) 필요

    # Backpropagation
    loss.backward()
    optimizer.step()

    return loss.item(), accuracy_function(logits, target, pad_id=sp.pad_id())

In [306]:
def train(model, dataloader, optimizer, loss_function, scheduler, num_epochs, device):
    model.to(device)

    for epoch in range(num_epochs):
        total_loss, total_acc = 0, 0
        for step, batch in enumerate(dataloader):
            loss, acc = train_step(model, batch, optimizer, loss_function, device)
            total_loss += loss
            total_acc += acc

            # 일정 스텝마다 로그 출력
            if step % 100 == 0:
                print(f"[Epoch {epoch+1}, Step {step}] Loss: {loss:.4f}, Acc: {acc:.4f}")

            # 학습률 스케줄러 업데이트
            scheduler.step()

        avg_loss = total_loss / len(dataloader)
        avg_acc = total_acc / len(dataloader)
        print(f"Epoch {epoch+1} Completed - Avg Loss: {avg_loss:.4f}, Avg Acc: {avg_acc:.4f}")

In [307]:
#######################################
######### Hyper #######################
#######################################


# 손실 함수(Loss function) :레이블인 시퀀스에 패딩이 되어 있으므로, loss를 계산할 때 패딩 마스크를 적용
loss_function = nn.CrossEntropyLoss(ignore_index=sp.pad_id())
EPOCH = 30
# 커스텀 된 학습률(Learning rate)
def get_lr_lambda(d_model, warmup_steps=4000):
    d_model = float(d_model)
    def lr_lambda(step):
        # step은 0부터 시작하므로 +1로 보정
        step = step + 1
        return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    return lr_lambda

# Optimizer 정의
optimizer = optim.Adam(model.parameters(), betas=(0.9, 0.98), eps=1e-9)
optimizer = optim.Adam(model.parameters(), lr=1)
# 
# Scheduler 정의
scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda(D_MODEL, warmup_steps=4000))

def accuracy_function(y_pred, y_true, pad_id=0):
    """
    y_pred: (batch_size, seq_len, vocab_size)
    y_true: (batch_size, seq_len)
    """
    preds = y_pred.argmax(dim=-1)  # (batch_size, seq_len)
    mask = (y_true != pad_id)
    correct = (preds == y_true) & mask
    acc = correct.float().sum() / mask.float().sum()
    return acc

In [308]:
%%time

train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_function=loss_function,
    scheduler=scheduler,
    num_epochs=EPOCH,  # 원하는 에폭 수
    device=device
)

[Epoch 1, Step 0] Loss: 9.1660, Acc: 0.0022
[Epoch 1, Step 100] Loss: 8.1783, Acc: 0.1463
[Epoch 1, Step 200] Loss: 7.1603, Acc: 0.1483
[Epoch 1, Step 300] Loss: 6.3309, Acc: 0.2004
Epoch 1 Completed - Avg Loss: 7.4356, Avg Acc: 0.1397
[Epoch 2, Step 0] Loss: 6.1851, Acc: 0.2042
[Epoch 2, Step 100] Loss: 6.0714, Acc: 0.2040
[Epoch 2, Step 200] Loss: 5.7541, Acc: 0.2295
[Epoch 2, Step 300] Loss: 5.4762, Acc: 0.2500
Epoch 2 Completed - Avg Loss: 5.7880, Avg Acc: 0.2247
[Epoch 3, Step 0] Loss: 5.4202, Acc: 0.2327
[Epoch 3, Step 100] Loss: 5.3051, Acc: 0.2654
[Epoch 3, Step 200] Loss: 5.0056, Acc: 0.2913
[Epoch 3, Step 300] Loss: 5.1675, Acc: 0.2651
Epoch 3 Completed - Avg Loss: 5.2499, Avg Acc: 0.2647
[Epoch 4, Step 0] Loss: 4.8356, Acc: 0.2812
[Epoch 4, Step 100] Loss: 4.6967, Acc: 0.3188
[Epoch 4, Step 200] Loss: 4.7431, Acc: 0.2985
[Epoch 4, Step 300] Loss: 4.5345, Acc: 0.2996
Epoch 4 Completed - Avg Loss: 4.7805, Avg Acc: 0.2984
[Epoch 5, Step 0] Loss: 4.4322, Acc: 0.3512
[Epoch 5, St

### **테스트하기**

In [309]:
def decoder_inference(model, sentence, tokenizer, device='cpu'):
    START_TOKEN = tokenizer.bos_id()
    END_TOKEN = tokenizer.eos_id()
    SEP_TOKEN = tokenizer.piece_to_id("<sep>")
    MAX_LENGTH = 50

    # 전처리
    sentence = preprocess_sentence_kor(sentence)

    # 질문 토큰화
    q_ids = tokenizer.encode(sentence)

    # GPT 입력: <bos> 질문 <sep>
    input_ids = [START_TOKEN] + q_ids + [SEP_TOKEN]

    dec_input = torch.tensor([input_ids], dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        for _ in range(MAX_LENGTH):

            #  GPT는 dec_input만 넣는다
            logits = model(dec_input)  # (1, seq_len, vocab_size)

            # 마지막 토큰 예측
            last_logits = logits[:, -1, :]  # (1, vocab_size)
            predicted_id = torch.argmax(last_logits, dim=-1)  # (1,)

            # 종료 조건
            if predicted_id.item() == END_TOKEN:
                break

            # 다음 토큰 붙이기
            dec_input = torch.cat(
                [dec_input, predicted_id.unsqueeze(0)], dim=1
            )

    # 결과 반환
    output_sequence = dec_input.squeeze(0).tolist()
    return output_sequence

In [310]:
def sentence_generation(model, sentence, tokenizer, device='cpu'):
    output_ids = decoder_inference(model, sentence, tokenizer, device)

    decoded = tokenizer.decode(output_ids)

    # special token 제거
    decoded = decoded.replace("<bos>", "").replace("<eos>", "")

    # 답변만 추출
    if "<sep>" in decoded:
        decoded = decoded.split("<sep>")[-1]

    return decoded.strip()

- ### 생성된 질문 확인

In [311]:
sentence = "영화 볼래?"
sentence_generation(model, sentence, sp, device)

'영화 볼매하세요 .'

In [312]:
sentence = "카페갈래?"
sentence_generation(model, sentence, sp, device)

'같이 수다떨고 영화보고 하세요 .'

In [313]:
sentence = "게임하고싶당"
sentence_generation(model, sentence, sp, device)

'게임하세요 !'

In [314]:
sentence = "게임하자"
sentence_generation(model, sentence, sp, device)

'조심하세요 .'

In [315]:
sentence = "너무 화가나"
sentence_generation(model, sentence, sp, device)

'이별의 빈자리가 느껴지으면 좋겠네요 .'

In [316]:
sentence = "고민이 있어"
sentence_generation(model, sentence, sp, device)

'생각보다 많은 시간이 지났네요 .'

## 회고

![nn](model_summary.png)

- GPT 논문(https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) 을 이해하고 Decoder-Only 인 구조를 따라해보는 시간이었다
- Encoder와 Encoder-Decoder Attention을 제거하면 논문에서 제시한 구조가 된다
- Input transformation을 해줘야한다. Classification, Entailment, Similarity, Multiple Choice을 방법이 제시되었다.
- 본 코드에서 넣은 input transformation은 Entailment으로 생각하고 SOS + Q + Delim + A + EOS형식으로 넣어보았다
-  '너무 화가나' 가 제일 문맥상 맞지 않은 답을 내놓는 것 같다. 이전 TASK에서는 생각하지 못했었는데, 감정적인 input이 들어오면 이상한 답이 나오는게 아닐까 생각을 해보았다. 감정을 표현하는 label도 같이 학습이 되었다면 잘 나왔을까 고민을 해보았다
-  '게임하고싶당!' 에 '게임하세요 !'는 이전의 기본 Transformer 했을 때도 같이 나왔는데 확률적으로 이 값만 집중해서 attention 해서 그런 것 같다. Attention 논문 sup 그림처럼 어떤 단어에 집중했는지 시각적으로 볼 수 있었으면 쉽게 확인했을 것 같다